# 06 — Short-Term & Long-Term Memory

## Learning requirements
- phân biệt thread-scoped state và cross-thread memory;
- dùng LangGraph checkpointer với LangChain agent;
- hiểu production checkpointer cần durable database;
- thiết kế namespace để user A không leak sang user B;
- hiểu semantic / episodic / procedural memory.

```text
Short-term memory = current thread/session
Long-term memory  = persisted across threads/sessions
```

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))
from src.providers import get_chat_model

checkpointer = InMemorySaver()
agent = create_agent(
    model=get_chat_model(),
    tools=[],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "demo-thread-1"}}

agent.invoke(
    {"messages": [{"role": "user", "content": "My project is called LCSP."}]},
    config,
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my project called?"}]},
    config,
)
print(result["messages"][-1].content)

## Thread isolation test

Cùng agent, đổi `thread_id`. Agent không được magically biết conversation của thread trước trừ khi bạn cấp long-term store.

In [ ]:
other = {"configurable": {"thread_id": "demo-thread-2"}}
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my project called?"}]},
    other,
)
print(result2["messages"][-1].content)

## Long-term memory mental model

LangGraph Store lưu JSON theo `(namespace, key)`.

Ví dụ:
```text
namespace = ("users", user_id, "preferences")
key       = "profile"
```

Namespace là security/data-isolation design decision, không phải tiện tay đặt string.

Memory taxonomy:
- **semantic**: facts/preferences;
- **episodic**: previous experiences/events;
- **procedural**: reusable rules/how-to (skills là một dạng procedural memory).

## Production exercise

1. Thay `InMemorySaver` bằng Postgres checkpointer.
2. Restart process và chứng minh thread resume.
3. Tạo long-term store với namespace chứa `tenant_id` + `user_id`.
4. Viết tests chứng minh user A không đọc memory user B.
5. Thử summarization khi message history dài.

## Required output
- short-term memory demo;
- namespace design diagram;
- isolation tests;
- note giải thích tại sao conversation history không đồng nghĩa long-term memory.

## Done criteria
Memory lifecycle và access scope được thiết kế explicit.